# Support Ticket Auto-Tagging Experiments

This notebook contains experiments and analysis for the support ticket classification project.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Setup complete!")

## 2. Data Exploration

In [ ]:
# Load processed data
train_df = pd.read_csv('../data/processed/train.csv')
val_df = pd.read_csv('../data/processed/val.csv')
test_df = pd.read_csv('../data/processed/test.csv')

print(f"Train set: {len(train_df)} samples")
print(f"Validation set: {len(val_df)} samples")
print(f"Test set: {len(test_df)} samples")

# Display sample
train_df.head()

In [ ]:
# Category distribution
plt.figure(figsize=(14, 6))
category_counts = train_df['category'].value_counts().head(20)
category_counts.plot(kind='bar', color='steelblue', alpha=0.8)
plt.title('Top 20 Categories Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"\nTotal unique categories: {train_df['category'].nunique()}")

In [ ]:
# Text length distribution
train_df['text_length'] = train_df['ticket_text'].str.len()

plt.figure(figsize=(12, 5))
plt.hist(train_df['text_length'], bins=50, color='coral', alpha=0.7, edgecolor='black')
plt.title('Ticket Text Length Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Text Length (characters)')
plt.ylabel('Frequency')
plt.axvline(train_df['text_length'].mean(), color='red', linestyle='--', label=f'Mean: {train_df["text_length"].mean():.0f}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Average text length: {train_df['text_length'].mean():.2f} characters")
print(f"Median text length: {train_df['text_length'].median():.2f} characters")

## 3. Test Zero-Shot Classification

In [ ]:
from src.zero_shot_classifier import ZeroShotClassifier

# Initialize classifier
zero_shot = ZeroShotClassifier()
zero_shot.load_categories()
zero_shot.load_model()

# Test on sample
sample_ticket = "My laptop battery drains very quickly and overheats."
predictions = zero_shot.predict_single(sample_ticket, top_k=3)

print(f"Ticket: {sample_ticket}\n")
print("Predictions:")
for i, (label, score) in enumerate(predictions, 1):
    print(f"  {i}. {label}: {score:.4f}")

## 4. Test Few-Shot Classification

In [ ]:
from src.few_shot_classifier import FewShotClassifier

# Initialize classifier
few_shot = FewShotClassifier(n_examples=3)
few_shot.load_categories()
few_shot.load_examples()
few_shot.load_model()

# Test on sample
predictions = few_shot.predict_single(sample_ticket, top_k=3)

print(f"Ticket: {sample_ticket}\n")
print("Predictions:")
for i, (label, score) in enumerate(predictions, 1):
    print(f"  {i}. {label}: {score:.4f}")

## 5. Compare Model Results

In [ ]:
import json

# Load results
results = {}
results_dir = Path('../results')

for result_file in results_dir.glob('*_results.json'):
    with open(result_file, 'r') as f:
        model_name = result_file.stem.replace('_results', '')
        results[model_name] = json.load(f)

# Display results
results_df = pd.DataFrame(results).T
print("Model Comparison:")
results_df

In [ ]:
# Visualize comparison
if 'accuracy_top1' in results_df.columns:
    plt.figure(figsize=(10, 6))
    results_df[['accuracy_top1', 'accuracy_top3']].plot(kind='bar', alpha=0.8)
    plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
    plt.xlabel('Model')
    plt.ylabel('Accuracy')
    plt.xticks(rotation=45)
    plt.legend(['Top-1 Accuracy', 'Top-3 Accuracy'])
    plt.tight_layout()
    plt.show()

## 6. Error Analysis

In [ ]:
# Analyze misclassifications
from src.predict import TicketPredictor

try:
    predictor = TicketPredictor()
    
    # Get predictions on test set
    test_sample = test_df.head(20)
    
    errors = []
    for idx, row in test_sample.iterrows():
        predictions = predictor.predict(row['ticket_text'], top_k=1)
        predicted_label = predictions[0][0]
        
        if predicted_label != row['category']:
            errors.append({
                'text': row['ticket_text'],
                'true': row['category'],
                'predicted': predicted_label
            })
    
    if errors:
        print(f"Found {len(errors)} misclassifications in sample:\n")
        for i, error in enumerate(errors[:5], 1):
            print(f"{i}. Text: {error['text'][:80]}...")
            print(f"   True: {error['true']}")
            print(f"   Predicted: {error['predicted']}\n")
    else:
        print("No errors found in sample!")
        
except Exception as e:
    print(f"Error analysis requires trained model: {e}")

## 7. Custom Predictions

In [ ]:
# Test your own tickets
custom_tickets = [
    "I can't log into my account after changing my password",
    "The software keeps freezing when I open multiple windows",
    "Need help configuring VPN on my work laptop"
]

try:
    predictor = TicketPredictor()
    
    for ticket in custom_tickets:
        print(f"Ticket: {ticket}")
        predictions = predictor.predict(ticket, top_k=3)
        print("Top 3 Tags:")
        for i, (tag, conf) in enumerate(predictions, 1):
            print(f"  {i}. {tag} ({conf:.4f})")
        print()
        
except Exception as e:
    print(f"Prediction requires trained model: {e}")

## Conclusion

This notebook demonstrates:
1. Data exploration and analysis
2. Zero-shot classification
3. Few-shot learning
4. Model comparison
5. Error analysis
6. Custom predictions

For production use, the fine-tuned model typically provides the best performance.